# Cleanup All Resources - GCP Dataplex Demo

This notebook orchestrates the cleanup of all resources by running individual cleanup notebooks in the correct order.

**Cleanup notebooks that will be executed:**

1. `06_cleanup_dataplex_metadata.ipynb` - Dataplex custom entries, entry types, entry groups
2. `03_cleanup_glossary_terms.ipynb` - Dataplex glossary, terms, categories
3. `01_cleanup_config_and_data.ipynb` - BigQuery aspects, aspect types, census_bureau_acs dataset
4. `02_cleanup_gcs_census.ipynb` - GCS bucket, census_uk_2021 dataset, BigQuery connection
5. `05_cleanup_firestore.ipynb` - Firestore data and database
6. `04_cleanup_cloudsql.ipynb` - CloudSQL instance (slowest, 5-10 min)

**⚠️  WARNING:**
- This operation cannot be undone
- All data will be permanently deleted
- Total execution time: approximately 10-15 minutes

**Required permissions:**
- `roles/bigquery.admin`
- `roles/storage.admin`
- `roles/cloudsql.admin`
- `roles/dataplex.admin`
- `roles/dataplex.catalogAdmin`
- `roles/datastore.owner`

## Section 1: Configuration & Setup

In [ ]:
# Configuration
import subprocess
import sys
import os
from pathlib import Path

# Get the cleanup directory path
CLEANUP_DIR = Path.cwd()

# Notebooks to execute in order
CLEANUP_NOTEBOOKS = [
    "06_cleanup_dataplex_metadata.ipynb",
    "03_cleanup_glossary_terms.ipynb",
    "01_cleanup_config_and_data.ipynb",
    "02_cleanup_gcs_census.ipynb",
    "05_cleanup_firestore.ipynb",
    "04_cleanup_cloudsql.ipynb"
]

print("📋 Cleanup Orchestrator Configuration:")
print(f"   Cleanup directory: {CLEANUP_DIR}")
print(f"   Number of notebooks: {len(CLEANUP_NOTEBOOKS)}")
print()
print("   Execution order:")
for i, notebook in enumerate(CLEANUP_NOTEBOOKS, 1):
    print(f"   {i}. {notebook}")

In [ ]:
# Verify jupyter nbconvert is available
print("🔍 Checking for jupyter nbconvert...")

try:
    result = subprocess.run(
        ["jupyter", "nbconvert", "--version"],
        capture_output=True,
        text=True
    )
    
    if result.returncode == 0:
        print(f"✅ jupyter nbconvert found: {result.stdout.strip()}")
    else:
        print("❌ jupyter nbconvert not found. Installing...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "nbconvert"])
        print("✅ jupyter nbconvert installed")
        
except FileNotFoundError:
    print("❌ jupyter command not found. Installing...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "jupyter", "nbconvert"])
    print("✅ jupyter and nbconvert installed")

## Section 2: Execute Cleanup Notebooks

Execute each cleanup notebook in order. Errors in individual notebooks will be reported but won't stop execution of subsequent notebooks.

In [ ]:
# Function to execute a notebook
def execute_notebook(notebook_name):
    """Execute a notebook and return success status."""
    notebook_path = CLEANUP_DIR / notebook_name
    
    if not notebook_path.exists():
        print(f"   ⚠️  Notebook not found: {notebook_name}")
        return False
    
    print(f"   Executing: {notebook_name}")
    print(f"   ⏳ Please wait...")
    print()
    
    try:
        result = subprocess.run(
            [
                "jupyter", "nbconvert",
                "--to", "notebook",
                "--execute",
                "--inplace",
                "--ExecutePreprocessor.timeout=1800",  # 30 minute timeout
                str(notebook_path)
            ],
            capture_output=True,
            text=True,
            cwd=str(CLEANUP_DIR)
        )
        
        if result.returncode == 0:
            print(f"   ✅ Completed: {notebook_name}")
            return True
        else:
            print(f"   ⚠️  Errors encountered in {notebook_name}")
            if result.stderr:
                print(f"   Error details: {result.stderr[:500]}")
            return False
            
    except Exception as e:
        print(f"   ❌ Failed to execute {notebook_name}: {str(e)[:200]}")
        return False

# Execute all notebooks
print("🚀 Starting cleanup execution...")
print("=" * 70)
print()

results = {}

for i, notebook in enumerate(CLEANUP_NOTEBOOKS, 1):
    print(f"📝 Step {i}/{len(CLEANUP_NOTEBOOKS)}: {notebook}")
    print("-" * 70)
    
    success = execute_notebook(notebook)
    results[notebook] = success
    
    print()
    print("=" * 70)
    print()

print("✅ All cleanup notebooks executed!")

## Section 3: Cleanup Summary

Review the results of all cleanup operations.

In [ ]:
# Summary
print()
print("=" * 70)
print("🎉 CLEANUP ORCHESTRATION COMPLETED!")
print("=" * 70)
print()
print("📊 Execution Summary:")
print()

successful = sum(1 for success in results.values() if success)
failed = len(results) - successful

for i, (notebook, success) in enumerate(results.items(), 1):
    status = "✅ Success" if success else "⚠️  Had errors"
    print(f"   {i}. {notebook:<45} {status}")

print()
print(f"   Total: {len(results)} notebooks")
print(f"   Successful: {successful}")
print(f"   With errors: {failed}")
print()

if failed > 0:
    print("⚠️  Some cleanup operations had errors. Check individual notebook outputs for details.")
    print("   You may need to manually verify or clean up remaining resources.")
else:
    print("✅ All cleanup operations completed successfully!")

print()
print("🔗 Verify cleanup in GCP Console:")
print("   BigQuery: https://console.cloud.google.com/bigquery")
print("   Cloud Storage: https://console.cloud.google.com/storage")
print("   CloudSQL: https://console.cloud.google.com/sql")
print("   Dataplex: https://console.cloud.google.com/dataplex")
print("   Firestore: https://console.cloud.google.com/firestore")

## Resources Deleted

The following resources should have been deleted:

### BigQuery
- Dataset: `census_bureau_acs` (278 tables)
- Dataset: `census_uk_2021`
- Connection: `census-biglake-connection`

### Cloud Storage
- Bucket: `{PROJECT_ID}-census-data`

### CloudSQL
- Instance: `census-demo-db`
- Database: `census_data`
- Table: `census_residence_type`

### Dataplex
- Aspect Types: `census-survey-metadata`, `data-governance-public`
- Aspects: Removed from all BigQuery table entries
- Glossary: `acs-demographics-glossary`
- Entry: `cloudsql-census-demo-db-census-data-census-residence-type-copy-manual`
- Entry Type: `cloudsql-table`
- Entry Group: `cloudsql-entries`

### Firestore
- Database: `(default)`
- Collection: `datasets` (all documents and subcollections)